In [1]:
import os
import re
import fitz
import numpy as np

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from pymongo import MongoClient, ReplaceOne

c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PDF_PATH = r"./comparison_summary_BNSS_to_CrPC.pdf"

DB_NAME = "test_db"

COLLECTION_NAME = "mapping"

NEW_LAW = "BNSS"
OLD_LAW = "CRPC"

VECTOR_INDEX_NAME = "test_mapping"

In [3]:
pdf = fitz.open(PDF_PATH)

print("Pages:", len(pdf))

Pages: 38


In [4]:
def clean_cell(value):

    if value is None:
        return ""

    value = str(value)

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()

In [5]:
def extract_mapping_rows(pdf):

    rows = []

    current_row = None

    for page_number, page in enumerate(
        pdf,
        start=1
    ):

        tables = page.find_tables()

        if not tables.tables:
            continue

        for table in tables.tables:

            extracted_rows = table.extract()

            for raw_row in extracted_rows:

                cells = [
                    clean_cell(cell)
                    for cell in raw_row
                ]

                # Skip table header
                combined = " ".join(
                    cells
                ).lower()

                if (
                    "bnss" in combined
                    and "crpc" in combined
                    and "summary" in combined
                ):
                    continue

                # Some pages are detected as
                # >4 columns because of broken
                # table borders.
                if len(cells) >= 4:

                    bnss_section = cells[0]

                    subject = cells[1]

                    crpc_section = cells[2]

                    summary = " ".join(
                        cell
                        for cell in cells[3:]
                        if cell
                    )

                else:

                    cells += (
                        [""] * (
                            4 - len(cells)
                        )
                    )

                    (
                        bnss_section,
                        subject,
                        crpc_section,
                        summary
                    ) = cells[:4]

                # ---------------------------
                # New table row
                # ---------------------------

                if bnss_section:

                    if current_row:

                        rows.append(
                            current_row
                        )

                    current_row = {

                        "page":
                            page_number,

                        "new_section":
                            bnss_section,

                        "subject":
                            subject,

                        "old_section_raw":
                            crpc_section,

                        "summary":
                            summary
                    }

                # ---------------------------
                # Continuation of summary
                # ---------------------------

                elif (
                    current_row
                    and summary
                ):

                    current_row[
                        "summary"
                    ] += " " + summary

    if current_row:

        rows.append(
            current_row
        )

    return rows

In [6]:
mapping_rows = extract_mapping_rows(
    pdf
)

print(
    "Mapping rows:",
    len(mapping_rows)
)

Mapping rows: 558


In [7]:
def extract_section_refs(
    text
):

    if not text:
        return []

    text = text.strip()

    if text == "-":
        return []

    refs = re.findall(
        r"\b\d{1,3}[A-Z]?\b",
        text.upper()
    )

    # remove duplicates
    refs = list(
        dict.fromkeys(refs)
    )

    return refs

In [8]:
print(
    extract_section_refs(
        "41 & 41A"
    )
)

['41', '41A']


In [9]:
for row in mapping_rows:

    row["old_sections"] = (
        extract_section_refs(
            row["old_section_raw"]
        )
    )

In [10]:
for row in mapping_rows[:10]:

    print(
        row["new_section"],
        "->",
        row["old_sections"]
    )

1 -> ['1']
2 -> ['2']
2(1)(a) -> []
2(1)(b) -> []
2(1)(c) -> ['2']
2(1)(d) -> []
2(1)(e) -> []
2(1)(f) -> ['2']
2(1)(g) -> ['2']
2(1)(h) -> ['2']


In [11]:
def detect_mapping_type(
    summary,
    old_sections
):

    text = (
        summary.lower()
    )

    if not old_sections:

        return "newly_added"

    if (
        "newly added" in text
    ):
        return "newly_added"

    if (
        "no change" in text
        or text.strip() == "ditto."
    ):
        return "no_change"

    if (
        "excluded" in text
        and "added" not in text
    ):
        return "removed_or_excluded"

    return "modified"

In [12]:
for row in mapping_rows:

    row["mapping_type"] = (
        detect_mapping_type(
            row["summary"],
            row["old_sections"]
        )
    )

In [13]:
def build_mapping_text(
    row
):

    old_section_text = (
        row["old_section_raw"]
        if row["old_section_raw"]
        else "No corresponding CrPC section"
    )

    return f"""
New Law: Bharatiya Nagarik Suraksha Sanhita, 2023 (BNSS)
New Section: {row['new_section']}

Old Law: Code of Criminal Procedure, 1973 (CrPC)
Old Section: {old_section_text}

Subject:
{row['subject']}

Comparison:
{row['summary']}
""".strip()

In [14]:
print(
    build_mapping_text(
        mapping_rows[0]
    )
)

New Law: Bharatiya Nagarik Suraksha Sanhita, 2023 (BNSS)
New Section: 1

Old Law: Code of Criminal Procedure, 1973 (CrPC)
Old Section: 1

Subject:
Shorttitle extentand , commencement .

Comparison:
ItcontainstheActname BharatiyaNagarikSuraksha , Sanhita 2023 andprovisionsregardingthecommencement , , date .


In [15]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "Embedding dimension:",
    model.get_sentence_embedding_dimension()
)

print(
    "Max sequence:",
    model.max_seq_length
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2340.99it/s]


Embedding dimension: 384
Max sequence: 256


C:\Users\Himanshu\AppData\Local\Temp\ipykernel_19900\3794163349.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


In [22]:
def token_count(text):

    return len(
        model.tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )

In [23]:
def split_by_tokens(text, max_tokens, overlap=20):

    token_ids = model.tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    if len(token_ids) <= max_tokens:
        return [text]

    chunks = []

    step = max_tokens - overlap

    for start in range(0, len(token_ids), step):

        part_ids = token_ids[
            start:start + max_tokens
        ]

        part = model.tokenizer.decode(
            part_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        chunks.append(part)

        if start + max_tokens >= len(token_ids):
            break

    return chunks

In [24]:
def create_mapping_chunks(mapping_rows):

    chunks = []

    MAX_TOKENS = model.max_seq_length

    for row in mapping_rows:

        header = f"""
New Law: BNSS
New Section: {row['new_section']}

Old Law: CrPC
Old Section: {row['old_section_raw']}

Subject: {row['subject']}

Comparison:
""".strip()

        summary = row["summary"].strip()

        # Calculate available tokens for summary
        header_tokens = len(
            model.tokenizer.encode(
                header,
                add_special_tokens=False
            )
        )

        special_tokens = (
            model.tokenizer.num_special_tokens_to_add(
                pair=False
            )
        )

        available_tokens = (
            MAX_TOKENS
            - header_tokens
            - special_tokens
            - 12
        )

        if available_tokens <= 0:
            raise ValueError(
                f"Header too long for section "
                f"{row['new_section']}"
            )

        # Keep normal mapping rows intact.
        # Split only unusually long summaries.
        if token_count(header + "\n" + summary) <= MAX_TOKENS:

            parts = [summary]

        else:

            parts = split_by_tokens(
                summary,
                max_tokens=available_tokens,
                overlap=20
            )

        for index, part in enumerate(parts):

            chunk_text = (
                header + "\n" + part
            ).strip()

            # Final safety check
            if token_count(chunk_text) > MAX_TOKENS:
                raise ValueError(
                    f"Chunk exceeds token limit: "
                    f"BNSS Section {row['new_section']}, "
                    f"Chunk {index}, "
                    f"Tokens {token_count(chunk_text)}"
                )

            chunks.append(
                Document(

                    page_content=chunk_text,

                    metadata={

                        "new_law": "BNSS",

                        "old_law": "CRPC",

                        "new_section":
                            row["new_section"],

                        "old_section":
                            row["old_section_raw"],

                        "old_sections":
                            row["old_sections"],

                        "subject":
                            row["subject"],

                        "mapping_type":
                            row["mapping_type"],

                        "page":
                            row["page"],

                        "chunk_index": index,

                        "document_type":
                            "law_mapping",

                        "source":
                            os.path.basename(PDF_PATH)
                    }
                )
            )

    return chunks

In [25]:
chunks = create_mapping_chunks(
    mapping_rows
)

invalid_chunks = []

max_tokens = 0

for chunk in chunks:

    count = token_count(
        chunk.page_content
    )

    max_tokens = max(
        max_tokens,
        count
    )

    if count > model.max_seq_length:

        invalid_chunks.append({

            "section":
                chunk.metadata["new_section"],

            "chunk":
                chunk.metadata["chunk_index"],

            "tokens": count
        })


print("Total mapping chunks:", len(chunks))

print("Maximum tokens:", max_tokens)

print("Invalid chunks:", len(invalid_chunks))

print("Invalid examples:", invalid_chunks[:5])

assert len(invalid_chunks) == 0, (
    "Some chunks exceed the model token limit!"
)

Total mapping chunks: 559
Maximum tokens: 244
Invalid chunks: 0
Invalid examples: []


In [26]:
invalid_chunks = []

max_tokens = 0


for chunk in chunks:

    count = token_count(
        chunk.page_content
    )

    max_tokens = max(
        max_tokens,
        count
    )

    if (
        count
        > model.max_seq_length
    ):

        invalid_chunks.append({

            "new_section":
                chunk.metadata[
                    "new_section"
                ],

            "chunk_index":
                chunk.metadata[
                    "chunk_index"
                ],

            "tokens":
                count
        })


print(
    "Max tokens:",
    max_tokens
)

print(
    "Invalid chunks:",
    len(invalid_chunks)
)

print(
    invalid_chunks[:10]
)


assert (
    len(invalid_chunks)
    == 0
)

Max tokens: 244
Invalid chunks: 0
[]


In [27]:
embedding_texts = [

    chunk.page_content

    for chunk in chunks
]

In [28]:
embeddings = model.encode(

    embedding_texts,

    batch_size=32,

    normalize_embeddings=True,

    show_progress_bar=True
)

Batches: 100%|██████████| 18/18 [00:07<00:00,  2.29it/s]


In [29]:
print(
    embeddings.shape
)

assert (
    len(embeddings)
    == len(chunks)
)

(559, 384)


In [30]:
MONGO_URI = os.getenv(
    "MONGODB_URI"
)

client = MongoClient(
    MONGO_URI
)

db = client[
    "test_db"
]

collection = db[
    "mapping"
]


print(
    client.admin.command(
        "ping"
    )
)

{'ok': 1}


In [31]:
mongo_documents = []


for i, chunk in enumerate(
    chunks
):

    new_section = (
        chunk.metadata[
            "new_section"
        ]
    )

    chunk_index = (
        chunk.metadata[
            "chunk_index"
        ]
    )


    mongo_documents.append({

        "id":
            f"bnss-crpc-"
            f"{new_section}-"
            f"{chunk_index}",

        "values":
            embeddings[i]
            .tolist(),

        "metadata": {

            **chunk.metadata,

            "text":
                chunk.page_content
        }
    })


print(
    "Prepared:",
    len(mongo_documents)
)

Prepared: 559


In [32]:
collection.create_index(
    "id",
    unique=True
)

'id_1'

In [33]:
operations = [

    ReplaceOne(

        {
            "id":
                document[
                    "id"
                ]
        },

        document,

        upsert=True
    )

    for document
    in mongo_documents
]

In [34]:
collection.bulk_write(
    operations
)

BulkWriteResult({'writeErrors': [], 'writeConcernErrors': [], 'nInserted': 0, 'nUpserted': 559, 'nMatched': 0, 'nModified': 0, 'nRemoved': 0, 'upserted': [{'index': 0, '_id': ObjectId('6afc83be8c3c43530658d7e0')}, {'index': 1, '_id': ObjectId('6afc83be8c3c43530658d7e1')}, {'index': 2, '_id': ObjectId('6afc83be8c3c43530658d7e2')}, {'index': 3, '_id': ObjectId('6afc83be8c3c43530658d7e3')}, {'index': 4, '_id': ObjectId('6afc83be8c3c43530658d7e4')}, {'index': 5, '_id': ObjectId('6afc83be8c3c43530658d7e5')}, {'index': 6, '_id': ObjectId('6afc83be8c3c43530658d7e6')}, {'index': 7, '_id': ObjectId('6afc83be8c3c43530658d7e7')}, {'index': 8, '_id': ObjectId('6afc83be8c3c43530658d7e8')}, {'index': 9, '_id': ObjectId('6afc83be8c3c43530658d7e9')}, {'index': 10, '_id': ObjectId('6afc83be8c3c43530658d7ea')}, {'index': 11, '_id': ObjectId('6afc83be8c3c43530658d7eb')}, {'index': 12, '_id': ObjectId('6afc83be8c3c43530658d7ec')}, {'index': 13, '_id': ObjectId('6afc83be8c3c43530658d7ed')}, {'index': 14, '

In [35]:
print(
    "BNSS-CrPC records:",
    collection.count_documents(

        {
            "metadata.new_law":
                "BNSS",

            "metadata.old_law":
                "CRPC"
        }
    )
)

BNSS-CrPC records: 559


In [36]:
VECTOR_INDEX_NAME = (
    "test_mapping"
)

In [37]:
def mongo_mapping_search(
    query,
    k=5
):

    query_vector = model.encode(
        [query],
        normalize_embeddings=True
    )[0].tolist()


    pipeline = [

        {
            "$vectorSearch": {

                "index":
                    VECTOR_INDEX_NAME,

                "path":
                    "values",

                "queryVector":
                    query_vector,

                "numCandidates":
                    max(
                        100,
                        k * 20
                    ),

                "limit":
                    k,

                "filter": {

                    "metadata.new_law":
                        "BNSS",

                    "metadata.old_law":
                        "CRPC"
                }
            }
        },

        {
            "$project": {

                "_id": 0,

                "id": 1,

                "metadata": 1,

                "score": {
                    "$meta":
                        "vectorSearchScore"
                }
            }
        }
    ]


    return list(
        collection.aggregate(
            pipeline
        )
    )

In [38]:
def find_by_bnss_section(
    section
):

    return list(
        collection.find(

            {
                "metadata.new_law":
                    "BNSS",

                "metadata.old_law":
                    "CRPC",

                "metadata.new_section":
                    str(section)
            },

            {
                "_id": 0
            }
        )
    )

In [40]:
results = find_by_bnss_section(
    "51"
)

for result in results:

    print(
        result[
            "metadata"
        ]["text"]
    )

New Law: BNSS
New Section: 51

Old Law: CrPC
Old Section: 53

Subject: Examinationofaccused by medical practitioneratthe requestofpoliceofficer .

Comparison:
From sub -section (1) "not below the rank of sub -inspector" , isexcluded andsubsection(3)isnewlyaddedregardingthe , forwardingofexaminationreportswithout delaybya medicalpractitionertotheinvestigator Inexplanation(b) . , the"definedinclause(h) ofSection2ofthe IndianMedical Council Act 1956(102of1956)"isreplacedby , "recognisedundertheNationalMedicalCommissionAct , 2019(30of2019)"and "theNationalMedicalRegister"is , alsoaddedas anoptionaltotheStateMedicalRegister .
